# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [26]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania: 3.55s


In [28]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania (wątki): 0.77s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [20]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [8]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 8 procesach (rdzeniach)...
Zakończono w czasie 2.66s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [ ]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures
import logging

CAT_API_URL = "https://catfact.ninja/fact"

# zad 1 - sekwencyjnie 
logging.basicConfig(level=logging.INFO, format='%(message)s')

def download_site(url):
    """
    Pobiera treść strony i wyciąga fakty o kotach.
    """
    facts_from_site: list[str] = []
    try:
        response = requests.get(url).json().get('fact')
        fact = response.strip()
        facts_from_site.append(fact)
        logging.info(f"Pobrano: {fact[:50]}...")

    except Exception as e:
        logging.error(f"Nie udało się pobrać {url}: {e}")
        
    return facts_from_site

def download_seq():
    logging.info(f"--- ROZPOCZĘCIE POBIERANIA SEKWENCYJNEGO ---")
    start_time = time.time()
    facts = []

    for i in range(20):
        logging.info(f"Pobieranie faktu {i+1}...")
        fact = download_site(CAT_API_URL)
        facts.append(fact)

    end_time = time.time()
    factsNumber = len(facts)
    duration = end_time - start_time
    
    logging.info(f"--- ZAKOŃCZONO ---")
    logging.info(f"Łącznie pobrano faktow: {factsNumber}")
    logging.info(f"\nCzas wykonania sekwencyjnego: {duration:.2f} sekund")
#download_seq()
#Czas wykonania sekwencyjnego: 14.46 sekund

def download_threaded():
    logging.info(f"--- ROZPOCZĘCIE POBIERANIA WIELOWĄTKOWE ---")
    start_time = time.time()
    urls = [CAT_API_URL] * 20
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, urls))
    
    all_facts = [fact for sublist in results for fact in sublist]
    factsNumber = len(all_facts)
    end_time = time.time()
    duration = end_time - start_time
    
    logging.info(f"--- ZAKOŃCZONO ---")
    logging.info(f"Łącznie pobrano faktow: {factsNumber}")
    logging.info(f"\nCzas wykonania wielowątkowego: {duration:.2f} sekund")

#download_threaded()
#Czas wykonania wielowątkowego: 4.43 sekund

--- ROZPOCZĘCIE POBIERANIA WIELOWĄTKOWE ---
  [OK] Pobrano: While many cats enjoy milk, it will give some cats...
  [OK] Pobrano: A cat has two vocal chords, and can make over 100 ...
  [OK] Pobrano: A cat will tremble or shiver when it is extreme pa...
  [OK] Pobrano: The life expectancy of cats has nearly doubled ove...
  [OK] Pobrano: The cat who holds the record for the longest non-f...
  [OK] Pobrano: The longest living cat on record according to the ...
  [OK] Pobrano: Cheetahs do not roar, as the other big cats do. In...
  [OK] Pobrano: Cats are extremely sensitive to vibrations. Cats a...
  [OK] Pobrano: The first true cats came into existence about 12 m...
  [OK] Pobrano: Cats, just like people, are subject to asthma. Dus...
  [OK] Pobrano: A cat can spend five or more hours a day grooming ...
  [OK] Pobrano: Contrary to popular belief, the cat is a social an...
  [OK] Pobrano: Tests done by the Behavioral Department of the Mus...
  [OK] Pobrano: The ability of a cat to find i

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [7]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time
import logging

logging.basicConfig(level=logging.INFO, format='%(message)s')

def producent_consumers():
    buffer = queue.Queue(maxsize=10)
    buffer_p = queue.Queue(maxsize=10)
    buffer_n = queue.Queue(maxsize=10)
    num_items = 20

    def producent():
        for i in range(1, num_items + 1):
            item = i
            buffer.put(item)
            logging.info(f"Wyprodukowano: {item}")
            time.sleep(0.1)
        buffer.put(None)

    def dispatcher():
        while True:
            item = buffer.get()
            if item is None:
                buffer_p.put(None)
                buffer_n.put(None)
                break
            if item % 2 == 0:
                buffer_p.put(item)
            else:
                buffer_n.put(item)

    def consumer_p():
        while True:
            item = buffer_p.get()
            if item is None:
                break
            logging.info(f"Konsument 1: {item}")
            buffer_p.task_done()
            time.sleep(0.1)

    def consumer_n():
        while True:
            item = buffer_n.get()
            if item is None:
                break
            logging.info(f"Konsument 2: {item}")
            buffer_n.task_done()
            time.sleep(0.1)

    prod_thread = threading.Thread(target=producent)
    dysp_thread =threading.Thread(target=dispatcher)
    cons1_thread = threading.Thread(target=consumer_p)
    cons2_thread = threading.Thread(target=consumer_n)

    prod_thread.start()
    cons1_thread.start()
    cons2_thread.start()
    dysp_thread.start()

    prod_thread.join()
    dysp_thread.join()
    cons1_thread.join()
    cons2_thread.join()

producent_consumers()

Wyprodukowano: 1
Konsument 2: 1
Wyprodukowano: 2
Konsument 1: 2
Wyprodukowano: 3
Konsument 2: 3
Wyprodukowano: 4
Konsument 1: 4
Wyprodukowano: 5
Konsument 2: 5
Wyprodukowano: 6
Konsument 1: 6
Wyprodukowano: 7
Konsument 2: 7
Wyprodukowano: 8
Konsument 1: 8
Wyprodukowano: 9
Konsument 2: 9
Wyprodukowano: 10
Konsument 1: 10
Wyprodukowano: 11
Konsument 2: 11
Wyprodukowano: 12
Konsument 1: 12
Wyprodukowano: 13
Konsument 2: 13
Wyprodukowano: 14
Konsument 1: 14
Wyprodukowano: 15
Konsument 2: 15
Wyprodukowano: 16
Konsument 1: 16
Wyprodukowano: 17
Konsument 2: 17
Wyprodukowano: 18
Konsument 1: 18
Wyprodukowano: 19
Konsument 2: 19
Wyprodukowano: 20
Konsument 1: 20


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [24]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def run_power_sum():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    num_range = range(1, 10001)

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, num_range)
        #starmap gdy więcej argumentów, map gdy jeden argument
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")
    #test
    #print(results[:5])

if __name__ == "__main__":
    run_power_sum()

Praca na 8 procesach (rdzeniach)...
Zakończono w czasie 0.80s.
